in this notebook we map data from 64 classes to 12 categories

In [5]:
import pandas as pd
import numpy as np

In [6]:


# 1. Define the 12 supercategories in order
SUPER_CATEGORIES = [
    'vegetation', 'terrain', 'sky', 'construction', 
    'vehicle', 'road', 'object', 'void', 'sign', 
    'human', 'water', 'animal'
]
cat_to_id = {cat: idx for idx, cat in enumerate(SUPER_CATEGORIES)}

# 2. Map fine-grained classes to the 12 supercategories (from attached image)
CLASS_TO_CATEGORY = {
    # vegetation
    'bush': 'vegetation', 'crops': 'vegetation', 'forest': 'vegetation', 
    'hedge': 'vegetation', 'high_grass': 'vegetation', 'leaves': 'vegetation', 
    'low_grass': 'vegetation', 'moss': 'vegetation', 'scenery_vegetation': 'vegetation', 
    'tree_crown': 'vegetation', 'tree_root': 'vegetation', 'tree_trunk': 'vegetation',
    
    # terrain
    'asphalt': 'terrain', 'cobble': 'terrain', 'gravel': 'terrain', 
    'snow': 'terrain', 'soil': 'terrain',
    
    # sky
    'sky': 'sky',
    
    # construction
    'bridge': 'construction', 'building': 'construction', 'container': 'construction', 
    'debris': 'construction', 'fence': 'construction', 'guard_rail': 'construction', 
    'tunnel': 'construction', 'wall': 'construction', 'wire': 'construction',
    
    # vehicle
    'bicycle': 'vehicle', 'bus': 'vehicle', 'car': 'vehicle', 'caravan': 'vehicle', 
    'heavy_machinery': 'vehicle', 'kick_scooter': 'vehicle', 'motorcycle': 'vehicle', 
    'on_rails': 'vehicle', 'trailer': 'vehicle', 'truck': 'vehicle', 
    'military_vehicle': 'vehicle',
    
    # road
    'bikeway': 'road', 'curb': 'road', 'pedestrian_crossing': 'road', 
    'rail_track': 'road', 'road_marking': 'road', 'sidewalk': 'road',
    
    # object
    'barrel': 'object', 'obstacle': 'object', 'pipe': 'object', 
    'pole': 'object', 'rock': 'object', 'street_light': 'object',
    
    # void
    'ego_vehicle': 'void', 'outlier': 'void', 'undefined': 'void',
    
    # sign
    'barrier_tape': 'sign', 'boom_barrier': 'sign', 'misc_sign': 'sign', 
    'road_block': 'sign', 'traffic_cone': 'sign', 'traffic_light': 'sign', 
    'traffic_sign': 'sign',
    
    # human
    'person': 'human', 'rider': 'human',
    
    # water
    'water': 'water',
    
    # animal
    'animal': 'animal'
}

# 3. Read CSV and sort by label_key
df = pd.read_csv('challenge_label_mapping.csv')
df['supercategory'] = df['class_name'].map(CLASS_TO_CATEGORY)
df['supercategory_id'] = df['supercategory'].map(cat_to_id)

# 4. Generate the 64-element array indexed strictly by label_key
mapping_array = df.sort_values('label_key')['supercategory_id'].to_numpy(dtype=np.int64)

print("Generated array shape:", mapping_array.shape)
print(mapping_array)

Generated array shape: (64,)
[ 7  8  1  1  6  0  6  5  7  5  8  5  4  4  9  4  0  0  0  8  4  5  5  1
  1  8  5  0  0  3  0  1  9 11  4  4  4  4  3  3  6  3  3  3  3  6  8  8
  8  4  0  0  0  2 10  3  7  4  3  0  6  6  0  4]


In [8]:
import torch
from PIL import Image

In [4]:


def preprocess_labelids_mask(mask_path):
    # Load 1-channel grayscale mask containing 0-63 IDs
    raw_mask = np.array(Image.open(mask_path))
    
    # Instant vectorized 0-63 -> 0-11 mapping
    mapped_mask = mapping_array[raw_mask]
    
    # Convert to PyTorch tensor for CrossEntropyLoss
    return torch.as_tensor(mapped_mask, dtype=torch.long)

In [ ]:
## TO CHANGE DIRS PATHES BEFORE USE

mask_path = "./data_short/masks/2022-07-22_flight__0000_1658492967230070008.png"
output_path = "./raw_12class_mask.png"

# 4. Process the mask
mapped_tensor = preprocess_labelids_mask(mask_path)

# Convert the PyTorch tensor back to an 8-bit unsigned integer NumPy array
# (Required format for saving standard images)
mapped_mask = mapped_tensor.numpy().astype(np.uint8)

# 5. Save directly as a raw 1-channel (grayscale) image
raw_image = Image.fromarray(mapped_mask, mode='L')
raw_image.save(output_path)

print(f"Successfully saved raw mask to: {output_path}")
print(f"Unique class IDs present in this image: {np.unique(mapped_mask)}")

Successfully saved raw mask to: ./raw_12class_mask.png
Unique class IDs present in this image: [0 1 2 3 6 7]


In [3]:
import os

In [9]:
## TO CHANGE DIRS PATHES BEFORE USE

input_dir = "./data_goose/test/masks"
output_dir = "./data_goose/test/masks_12class"
os.makedirs(output_dir, exist_ok=True)

for file in [f for f in os.listdir(input_dir) if f.endswith('.png')]:
    raw_mask = np.array(Image.open(os.path.join(input_dir, file)))
    mapped_mask = mapping_array[raw_mask].astype(np.uint8)
    Image.fromarray(mapped_mask, mode='L').save(os.path.join(output_dir, file))

In [10]:
len([f for f in os.listdir(output_dir) if f.endswith('.png')])

962